In [ ]:
import sys
!{sys.executable} -m pip install yt-dlp faster-whisper pyvis networkx google-generativeai imageio-ffmpeg scipy python-dotenv

In [ ]:
import os
import json
import warnings
import subprocess
import tempfile

import numpy as np
import scipy.io.wavfile as wav
import imageio_ffmpeg
import whisper
import yt_dlp
import networkx as nx
from pyvis.network import Network
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import display, HTML, IFrame

warnings.filterwarnings("ignore", message="FP16 is not supported on CPU")

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found. Put it in a .env file or export it in the environment before running this notebook.")
genai.configure(api_key=GEMINI_API_KEY)

generation_config = {"response_mime_type": "application/json"}
llm_model = genai.GenerativeModel("gemini-2.5-flash", generation_config=generation_config)

print("Loading Whisper model (small)...")
whisper_model = whisper.load_model("small")
whisper_model = whisper_model.to("cuda")
print("Whisper model ready!")

VIDEO_SOURCES = [
    {
        "url": "https://www.youtube.com/watch?v=uDS8AkTAcIU",
        "language": "Hindi-English (Hinglish)",
        "topic": "Huffman Coding / Data Compression",
        "output_prefix": "video1",
    },
    {
        "url": "https://www.youtube.com/watch?v=nbgtyBKn2tI&list=PLzjZaW71kMwQ-JABTOTypnpRk1BnD2Nx4",
        "language": "Hindi-English (Hinglish)",
        "topic": "Trees",
        "output_prefix": "video2",
    },
    {
        "url": "https://www.youtube.com/watch?v=VF8ORgF1OBo",
        "language": "Hindi-English (Hinglish)",
        "topic": "Nutrition in Human Beings",
        "output_prefix": "video3",
    },
    {
        "url": "https://www.youtube.com/watch?v=4IkIGhuhuxo&list=PLF_7kfnwLFCHamOEHrFDSVoI6wszQQheq&index=7",
        "language": "Hindi-English (Hinglish)",
        "topic": "Difference between Displacement and Distance",
        "output_prefix": "video4",
    },
    {
        "url": "https://www.youtube.com/watch?v=3tkcfvCNtM8",
        "language": "Hindi-English (Hinglish)",
        "topic": "Topological Sort Graph Algorithm",
        "output_prefix": "video5",
    },
]


def fetch_video_metadata(youtube_url):
    ydl_opts = {
        'quiet': True,
        'no_warnings': True,
        'skip_download': True,
        'noplaylist': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
    duration_secs = info.get('duration', 0) or 0
    minutes, seconds = divmod(int(duration_secs), 60)
    return {
        'title': info.get('title', 'Unknown Title'),
        'duration': f"{minutes}:{seconds:02d}",
    }


In [ ]:
def download_audio(youtube_url, output_filename="video_audio"):
    mp3_path = f"{output_filename}.mp3"
    if os.path.exists(mp3_path):
        print(f"Audio already exists, skipping download: {mp3_path}")
        return mp3_path

    print(f"Downloading audio from: {youtube_url}")
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': f'{output_filename}.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'ffmpeg_location': ffmpeg_path,
        'noplaylist': True,   
        'keepvideo': False,
        'quiet': True,
        'no_warnings': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

    # Remove any leftover intermediate video files (e.g. .mp4, .m4a, .webm)
    for ext in ("mp4", "m4a", "webm", "mkv", "mov"):
        leftover = f"{output_filename}.{ext}"
        if os.path.exists(leftover):
            os.remove(leftover)
            print(f"Removed intermediate file: {leftover}")

    print(f"Downloaded and saved as {mp3_path}")
    return mp3_path


In [ ]:
def transcribe_audio(audio_path, transcript_cache_path=None):
    if transcript_cache_path and os.path.exists(transcript_cache_path):
        print(f"Transcript cache found, skipping transcription: {transcript_cache_path}")
        with open(transcript_cache_path, "r", encoding="utf-8") as f:
            return f.read()

    print(f"Transcribing {audio_path}... ")
    ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name

    subprocess.run(
        [ffmpeg_exe, "-y", "-i", audio_path, "-ar", "16000", "-ac", "1", tmp_path],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    sample_rate, data = wav.read(tmp_path)
    os.unlink(tmp_path)  
    samples = data.astype(np.float32) / 32768.0

    result = whisper_model.transcribe(samples)
    text = result["text"]
    print("Transcription complete!")

    if transcript_cache_path:
        with open(transcript_cache_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"Transcript cached -> {transcript_cache_path}")

    return text


In [ ]:
def extract_pedagogical_flow(raw_transcript, language_declaration, topic=""):
  
    topic_hint = f" about {topic}" if topic else ""

    prompt = f"""You are an expert educational data scientist specialising in computer science \
pedagogy and Indian code-mixed academic content.

I will provide a raw, noisy transcript from a technical educational video{topic_hint}, \
delivered in {language_declaration}. The speaker freely mixes formal English terminology \
with colloquial Hindi/regional-language phrases (e.g., "tree mein jo upper wala node hai" \
should be standardised to "Root Node").

Your tasks:
1. **Concept Extraction** - Identify 8-15 core technical concepts explicitly taught or \
demonstrated in the video.
2. **Linguistic Standardisation** - Map every colloquial or code-mixed term to its canonical \
English academic name. List these explicitly in the "standardization_map" array.
3. **Prerequisite Mapping** - For every pair A -> B, add an edge ONLY if understanding A is \
*strictly necessary* before grasping B, based on the lecture's own pedagogical flow. \
Do NOT add speculative edges.
4. **Teaching Sequence** - List the concepts in the exact order the teacher introduces them \
in the lecture (pedagogical order).
5. **Metadata Enrichment** - Tag each concept with a difficulty level and knowledge type.

Return a single JSON object matching this EXACT schema - no markdown, no prose outside the JSON:
{{
  "video_language": "{language_declaration}",
  "video_topic": "{topic}",
  "standardization_map": [
    {{
      "colloquial": "<Colloquial / code-mixed Indic term as spoken>",
      "standard_term": "<Standard English academic term>"
    }}
  ],
  "teaching_sequence": [
    "<Concept name in the order the teacher introduces them>"
  ],
  "concepts": [
    {{
      "id": "c1",
      "name": "<Canonical English Term>",
      "description": "<One-sentence definition in plain English>",
      "difficulty": "<beginner|intermediate|advanced>",
      "type": "<definition|algorithm|data_structure|theorem|technique|example>"
    }}
  ],
  "prerequisites": [
    {{
      "source_id": "c1",
      "target_id": "c2",
      "rationale": "<One-sentence explanation of why c1 must precede c2>",
      "strength": "<strong|weak>"
    }}
  ]
}}

Transcript:
{raw_transcript}
"""

    print("Extracting concepts and mapping dependencies...")
    response = llm_model.generate_content(prompt)
    structured_data = json.loads(response.text)

    # Surface-level validation
    for key in ("concepts", "prerequisites", "standardization_map", "teaching_sequence"):
        if key not in structured_data:
            raise ValueError(f"LLM response is missing required key: '{key}'.")

    return structured_data


In [ ]:
def visualize_knowledge_graph(structured_data, output_html="knowledge_graph.html"):
    DIFFICULTY_COLORS = {
        "beginner":     "#4CAF50",   
        "intermediate": "#FF9800",   
        "advanced":     "#F44336",   
    }
    DEFAULT_COLOR = "#97C2FC"        

    G = nx.DiGraph()

    # Nodes
    for concept in structured_data["concepts"]:
        difficulty   = concept.get("difficulty", "").lower()
        concept_type = concept.get("type", "")
        color        = DIFFICULTY_COLORS.get(difficulty, DEFAULT_COLOR)
        tooltip = (
            f"<b>{concept['name']}</b><br>"
            f"<i>Type: {concept_type} | Level: {difficulty}</i><br><br>"
            f"{concept['description']}"
        )
        G.add_node(concept["id"], label=concept["name"], title=tooltip, color=color)

    # Edges
    for edge in structured_data["prerequisites"]:
        width = 3 if edge.get("strength") == "strong" else 1
        G.add_edge(
            edge["source_id"],
            edge["target_id"],
            title=edge.get("rationale", ""),
            width=width,
        )

    net = Network(
        notebook=False,
        directed=True,
        width="100%",
        height="650px",
        bgcolor="#1a1a2e",
        font_color="#e0e0e0",
        cdn_resources="in_line",
    )
    net.from_nx(G)
  # More outgoing edges-> larger node size
    for node in net.nodes:
        out_deg    = G.out_degree(node["id"])
        node["size"] = max(15, 10 + out_deg * 8)


    net.set_options("""
    {
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "UD",
          "sortMethod": "directed",
          "nodeSpacing": 180,
          "levelSeparation": 140
        }
      },
      "physics": { "enabled": false },
      "edges": {
        "arrows": { "to": { "enabled": true, "scaleFactor": 0.8 } },
        "smooth": { "type": "cubicBezier", "forceDirection": "vertical" }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "navigationButtons": true
      }
    }
    """)

    net.save_graph(output_html)
    print(f"Graph saved to {output_html}")
    return output_html


In [ ]:
REQUIRED_LLM_FIELDS = {"concepts", "prerequisites", "standardization_map", "teaching_sequence"}

all_results = {}  

for source in VIDEO_SOURCES:
    url    = source["url"]
    prefix = source["output_prefix"]
    lang   = source["language"]
    topic  = source["topic"]

    if url.startswith("REPLACE"):
        print(f"\n[WARN] Skipping {prefix} ({topic}) - URL not set.")
        continue

    print(f"\n{'='*60}")
    print(f"  Video : {topic}")
    print(f"  Lang  : {lang}")
    print(f"  URL   : {url}")
    print(f"{'='*60}")

    output_dir = prefix
    os.makedirs(output_dir, exist_ok=True)

    json_path       = os.path.join(output_dir, f"{prefix}_knowledge.json")
    transcript_path = os.path.join(output_dir, f"{prefix}_transcript.txt")
    graph_path      = os.path.join(output_dir, f"{prefix}_graph.html")
    audio_prefix    = os.path.join(output_dir, prefix)

    try:
        json_output = None

        if os.path.exists(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                cached_output = json.load(f)

            missing_fields = REQUIRED_LLM_FIELDS - set(cached_output.keys())
            if missing_fields:
                print(f"JSON cache is stale, missing: {sorted(missing_fields)}")
            else:
                print(f"Extraction cache found and complete, loading: {json_path}")
                json_output = cached_output

        if json_output is None:
            audio_file = download_audio(url, output_filename=audio_prefix)

            transcript = transcribe_audio(audio_file, transcript_cache_path=transcript_path)
            print(f"\nTranscript snippet:\n{transcript[:300]}...\n")

            json_output = extract_pedagogical_flow(transcript, lang, topic)

        print(f"Fetching live metadata for {prefix}...")
        yt_meta = fetch_video_metadata(url)
        json_output.update({
            "video_id": prefix,
            "video_title": yt_meta["title"],
            "video_url": url,
            "video_language": lang,
            "video_topic": topic,
            "duration": yt_meta["duration"],
        })

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(json_output, f, indent=2, ensure_ascii=False)
        print(f"JSON saved -> {json_path}")

        n_concepts = len(json_output["concepts"])
        n_edges = len(json_output["prerequisites"])
        print(f"Concepts: {n_concepts}  |  Prerequisite links: {n_edges}")

        if os.path.exists(graph_path):
            print(f"Graph cache found, displaying existing: {graph_path}")
        else:
            visualize_knowledge_graph(json_output, output_html=graph_path)

        all_results[prefix] = json_output

    except Exception as e:
        print(f"[ERROR] Error processing {prefix}: {e}")

print(f"\n\nPipeline complete - processed {len(all_results)} / {len(VIDEO_SOURCES)} video(s).")

In [ ]:
if not all_results:
    print("No videos processed yet - run the pipeline cell first.")
else:
    VIDEO_PALETTE = ["#97C2FC", "#FB7E81", "#FFA807", "#7DE27F", "#AD85E4"]

    master_G = nx.DiGraph()

    for idx, (prefix, data) in enumerate(all_results.items()):
        color = VIDEO_PALETTE[idx % len(VIDEO_PALETTE)]
        topic = next(
            (s["topic"] for s in VIDEO_SOURCES if s["output_prefix"] == prefix),
            prefix,
        )

        for concept in data["concepts"]:
            node_id = f"{prefix}__{concept['id']}"  
            difficulty   = concept.get("difficulty", "")
            concept_type = concept.get("type", "")
            tooltip = (
                f"<b>[{topic}]</b><br>"
                f"<b>{concept['name']}</b><br>"
                f"<i>{concept_type} | {difficulty}</i><br><br>"
                f"{concept['description']}"
            )
            master_G.add_node(
                node_id,
                label=concept["name"],
                title=tooltip,
                color=color,
                group=prefix,
            )

        for edge in data["prerequisites"]:
            master_G.add_edge(
                f"{prefix}__{edge['source_id']}",
                f"{prefix}__{edge['target_id']}",
                title=edge.get("rationale", ""),
                width=2 if edge.get("strength") == "strong" else 1,
            )

    master_net = Network(
        notebook=False,
        directed=True,
        width="100%",
        height="820px",
        bgcolor="#0d1117",
        font_color="#e0e0e0",
        cdn_resources="in_line",
    )
    master_net.from_nx(master_G)

    for node in master_net.nodes:
        out_deg      = master_G.out_degree(node["id"])
        node["size"] = max(12, 8 + out_deg * 6)

    master_net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -60,
          "centralGravity": 0.005,
          "springLength": 160,
          "springConstant": 0.08
        },
        "maxVelocity": 50,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": { "iterations": 250 }
      },
      "edges": {
        "arrows": { "to": { "enabled": true, "scaleFactor": 0.6 } },
        "smooth": { "type": "dynamic" }
      },
      "interaction": {
        "hover": true,
        "navigationButtons": true,
        "tooltipDelay": 100
      }
    }
    """)

    master_net.save_graph("master_knowledge_graph.html")
    print("Master knowledge graph saved -> master_knowledge_graph.html")

    legend_items = "".join(
        f'<span style="background:{VIDEO_PALETTE[i % len(VIDEO_PALETTE)]};'
        f'border-radius:4px;padding:2px 8px;margin:4px;color:#000;font-size:13px">'
        f'{src["output_prefix"]}: {src["topic"]}</span>'
        for i, src in enumerate(VIDEO_SOURCES)
        if src["output_prefix"] in all_results
    )
